# Notebook 4 — ML prediction (integrate ML into the platform)
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

Two Spark MLlib models:

1. **Traffic forecaster:** vehicles at each junction in the **next hour**. Features are the last hour, the same hour yesterday and last week, a 3-hour rolling mean, the hour, the weekday, the month and the junction.
   *Design choice:* traffic grows over time and tree models cannot predict above the range they were trained on, so every model learns the **change from the last hour**, and the last hour is added back.
2. **Air-quality soft sensor:** estimates true **CO** (mg/m³) from the 5 cheap metal-oxide sensors plus temperature and humidity. The expensive reference analyser is missing 18% of the time; the soft sensor fills those gaps in real time.

**Evaluation is time-based:** train on the past, choose hyper-parameters on a validation window, and report the untouched final window (test). A random split would leak the future into training.

| Model | Train | Validation | Test |
|---|---|---|---|
| Traffic | < 2017-02-01 | Feb 2017 | Mar–Jun 2017 |
| Air CO | < 2004-11-01 | Nov–Dec 2004 | Jan–Apr 2005 |

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pyspark.sql import functions as F
from src import config
from src.spark_utils import get_spark
spark = get_spark("NB04-ML")
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

In [ ]:
import json
from src import train_models
summary = train_models.run(spark)   # ~5-10 min: 3 model families x 2 settings x 2 models

In [ ]:
import pandas as pd
t = summary["traffic"]
rows = [{"model": b["model"], "params": b["params"], **{f"test_{k}": v for k, v in b["test"].items()}} for b in t["baselines"]]
rows += [{"model": c["model"], "params": c["params"], "val_rmse": c["val"]["rmse"], **{f"test_{k}": v for k, v in c["test"].items()}} for c in t["candidates"]]
print("Traffic forecaster: test window Mar-Jun 2017 (vehicles/hour)")
pd.DataFrame(rows)

In [ ]:
print("Best model per junction (test):")
pd.DataFrame(t["per_junction_test"]).T

In [ ]:
a = summary["air"]
print("Air CO soft sensor: test window Jan-Apr 2005 (mg/m3)")
pd.DataFrame([{"model": c["model"], "params": c["params"], "val_rmse": c["val"]["rmse"], **{f"test_{k}": v for k, v in c["test"].items()}} for c in a["candidates"]])

In [ ]:
print("Top features, traffic:", t["feature_importance"][:8])
print("Top features, air:    ", a["feature_importance"][:8])
print("\nAlert thresholds (90th percentile of training data):")
print(open(config.THRESHOLDS_JSON).read())

In [ ]:
from IPython.display import Image, display
for f in ["traffic_model_comparison.png", "traffic_actual_vs_predicted.png", "air_co_actual_vs_estimated.png"]:
    display(Image(filename=str(config.ML_OUT / f)))

### Sensor drift and how the live stream corrects it
The air test error grows over the months after training: cheap gas sensors drift as they age. The streaming job recalibrates online with the reference analyser's recent readings. Below, three strategies are replayed hour by hour over the test window. Each uses only readings from *before* the hour it corrects.

In [ ]:
import numpy as np
from pyspark.ml import PipelineModel
air_model = PipelineModel.load(config.p(config.AIR_MODEL_DIR))
test_air = spark.read.parquet(config.p(config.GOLD / "air_features")).filter(F.col("event_time") >= config.AIR_TEST_START)
p = air_model.transform(test_air).select("event_time", "co_gt", F.greatest(F.lit(0.0), "prediction").alias("est")).toPandas().sort_values("event_time")
y, est = p["co_gt"].to_numpy(), p["est"].to_numpy()

def replay(strategy, n=24):
    E, R, out = [], [], []
    for e, t in zip(est, y):
        k = 1.0
        if len(E) >= 6:
            if strategy == "ratio of sums (used)":
                k = min(max(sum(R[-n:]) / max(sum(E[-n:]), 1e-6), 0.5), 2.0)
            elif strategy == "subtract mean error":
                k = None; c = max(e - np.mean(np.array(E[-n:]) - np.array(R[-n:])), 0.0)
        out.append(e * k if k is not None else c)
        if not np.isnan(t):
            E.append(e); R.append(t)
    return np.array(out)

ok, limit = ~np.isnan(y), json.load(open(config.THRESHOLDS_JSON))["air_co_mg_m3"]
rows = []
for name in ["no correction", "subtract mean error", "ratio of sums (used)"]:
    c = est if name == "no correction" else replay(name)
    rows.append({"strategy": name, "MAE": round(np.abs(c - y)[ok].mean(), 3),
                 "false_alerts": int(((c > limit) & (y <= limit))[ok].sum()),
                 "missed_alerts": int(((c <= limit) & (y > limit))[ok].sum()),
                 "hours_pinned_to_0": int((c == 0).sum())})
pd.DataFrame(rows)

The saved `PipelineModel`s (imputer → one-hot encoder → assembler → model) are loaded by the streaming job, so the live data goes through **exactly the same preprocessing** as the training data.

> 📸 **Screenshot 6:** metrics tables and the three charts.

In [ ]:
spark.stop()